# Data collection , preprocessing and EDA



In [ ]:
API_KEYS = [
"XXX",
"XXX"
]

# Tracks which key is active and how many units it has used to manage 2 api keys
current_key_index = 0
units_used        = [0, 0]
SEARCH_COST       = 100
VIDEOS_COST       = 1
DAILY_LIMIT       = 9_500

In [ ]:
# colab access to drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# current folder
import os
print(os.getcwd())

In [ ]:
# Base path
BASE = "/content/drive/MyDrive/capstone"

In [ ]:
# creating folders for project

import os
os.makedirs(f"{BASE}/Project2data/raw",        exist_ok=True)
os.makedirs(f"{BASE}/Project2data/processed",  exist_ok=True)
os.makedirs(f"{BASE}/Project2data/thumbnails", exist_ok=True)
os.makedirs(f"{BASE}/Project2data/embeddings", exist_ok=True)
os.makedirs(f"{BASE}/models",          exist_ok=True)
os.makedirs(f"{BASE}/outputs",         exist_ok=True)

print(f"Working folder: {BASE}")
print("Folders created ✓")

In [ ]:
# installing dependenceis

import sys # cmnd line access to path and folders

# Xgboost
# scklit-learn
# shap
# scipy
# umap-learn
# opencv-python
# pillow
# pytesseract
# CLIP from OpenAI
# matpotlib & seaborn
# progress bar : tqdm
# isodate
# google-api-python-client
!pip install -q xgboost scikit-learn shap scipy isodate \
    opencv-python Pillow pytesseract \
    google-api-python-client tqdm \
    umap-learn matplotlib seaborn

# install CLIP library from Github reoisitory, which holds the model
!{sys.executable} -m pip install -q \
    git+https://github.com/openai/CLIP.git \
    torch torchvision

In [ ]:
import os
import time
import json
import requests
import pandas as pd
from PIL import Image
from io import BytesIO
from googleapiclient.discovery import build


# youtube resirces object

def build_youtube(key_index):
    return build("youtube", "v3", developerKey=API_KEYS[key_index])

youtube = build_youtube(current_key_index)   # start with key 1


# ── Switch to the next key when current one is close to the limit
def rotate_key_if_needed(cost):
    global current_key_index, youtube

    units_used[current_key_index] += cost    # add this call's cost to current key's total

    if units_used[current_key_index] >= DAILY_LIMIT:    # if current key is almost maxed out
        next_index = 1 - current_key_index              # flip between 0 and 1
        if units_used[next_index] < DAILY_LIMIT:        # only switch if the other key has quota left
            current_key_index = next_index              # switch to the other key
            youtube = build_youtube(current_key_index)  # rebuild the client with the new key
            print(f"🔁 Switched to API key {current_key_index + 1}")
        else:
            raise Exception("Both API keys have hit their daily quota. Resume tomorrow.")


BASE = "/content/drive/MyDrive/capstone"
print("Imports done ")
print(f"Base path: {BASE}")

OUTPUT_DIR = f"{BASE}/Project2data/thumbnails"           # thumbnails go into your existing thumbnails folder
CSV_PATH   = f"{BASE}/Project2data/raw/video_metadata.csv"  # metadata CSV goes into your raw folder



In [ ]:
# fetch youtube video thumbnails


# detect if video is a short
# cheks furation oand dimstion of thumbial
def is_short(duration_iso: str, width: int, height: int) -> bool:
    import re
    match = re.match(r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?', duration_iso)
    h = int(match.group(1) or 0)
    m = int(match.group(2) or 0)
    s = int(match.group(3) or 0)
    total_seconds = h * 3600 + m * 60 + s

    if width > 0 and height > 0:
        return total_seconds <= 60 and height > width  # use both if dimensions exist
    else:
        return total_seconds <= 60                     # fall back to duration only if no dimensions



# downloading thubnail
# saves thumbail image
def download_thumbnail(video_id: str, thumbnails: dict) -> dict:
  print("in download thubnial function")
  for quality in ["maxres", "standard", "high", "medium", "default"]:
    if quality in thumbnails:
      url = thumbnails[quality]["url"]
      w = thumbnails[quality].get("width", 0)
      h = thumbnails[quality].get("height", 0)
      try:
        resp = requests.get(url, timeout= 10)
        img = Image.open(BytesIO(resp.content))
        path = os.path.join(BASE, "Project2data", "thumbnails", f"{video_id}_{quality}.jpg")
        img.save(path)
        return {"url":url,"quality":quality,"width":w,"height":h, "path":path}
      except Exception as e:
        print(f"  ⚠ Thumbnail download failed for {video_id}: {e}")

  return {}

# requesting 50 ids
# calls api and requests it metdata
def fetch_batch(video_ids: list) -> list:
  rotate_key_if_needed(VIDEOS_COST)

  # getting video info like title, desc, channel name, duration, video quality, view count, like count and comment count
  response = youtube.videos().list(
    part="snippet,contentDetails,statistics",
    id=",".join(video_ids)
  ).execute()

  records = []

  for item in response.get("items", []):
    video_id = item["id"]
    snippet  = item["snippet"]
    stats    = item.get("statistics", {})
    details  = item["contentDetails"]
    thumbs   = snippet["thumbnails"]

    thumb_info = download_thumbnail(video_id, thumbs)

    # checking for shorts
    short = is_short(
            details["duration"],
            thumb_info.get("width", 0),
            thumb_info.get("height", 0)
        )

    records.append({
            "video_id":        video_id,
            "title":           snippet["title"],
            "channel":         snippet["channelTitle"],
            "channel_id":      snippet["channelId"],
            "published_at":    snippet["publishedAt"],
            "duration":        details["duration"],
            "is_short":        short,
            "view_count":      int(stats.get("viewCount", 0)),
            "like_count":      int(stats.get("likeCount", 0)),
            "comment_count":   int(stats.get("commentCount", 0)),
            "category_id":     snippet.get("categoryId"),
            "tags":            snippet.get("tags", []),
            "thumbnail_url":   thumb_info.get("url"),
            "thumbnail_quality": thumb_info.get("quality"),
            "thumbnail_width": thumb_info.get("width"),
            "thumbnail_height":thumb_info.get("height"),
            "thumbnail_path":  thumb_info.get("path"),
        })

  return records


# bulk fetching
# puts in in batches of 50
def bulk_fetch(video_ids: list, batch_size: int = 50, delay: float = 0.5) -> pd.DataFrame:

  all_records = []
  total = len(video_ids) # total number of vodeo id needed to process

  for i in range(0, total, batch_size): # loop in steps of 50
        batch = video_ids[i : i + batch_size] # slice next bathc of up to 50 IDs
        print(f"Fetching batch {i // batch_size + 1} / {-(-total // batch_size)}  ({len(batch)} videos)...")

        try:
            records = fetch_batch(batch) #fetches metadata and thumbnails for this batch
            all_records.extend(records)  # adds this batch's results to the master list
            print(f"  ✓ {len(records)} videos fetched  |  Total so far: {len(all_records)}")
        except Exception as e:
            print(f"  ✗ Batch failed: {e}")

        time.sleep(delay)  # be nice to the quota

  df = pd.DataFrame(all_records)
  df.to_csv(CSV_PATH, index=False)
  print(f"\n✅ Done! {len(df)} videos saved to video_metadata.csv")
  return df

# finds youtube video ID
def search_and_fetch(query: str, max_results: int = 200) -> pd.DataFrame:
    video_ids       = []
    next_page_token = None
    fetched         = 0

    print(f'Searching for: "{query}"')

    while fetched < max_results:
        n = min(50, max_results - fetched)

        rotate_key_if_needed(SEARCH_COST)

        search_resp = youtube.search().list(
            part="id",
            q=query,
            type="video",
            maxResults=n,
            pageToken=next_page_token,
            videoDuration="any"
        ).execute()

        for item in search_resp.get("items", []):
            video_ids.append(item["id"]["videoId"])

        fetched         += len(search_resp.get("items", []))
        next_page_token  = search_resp.get("nextPageToken")

        if not next_page_token:
            break
        time.sleep(0.3)

    print(f"Found {len(video_ids)} video IDs. Fetching metadata + thumbnails...\n")
    return bulk_fetch(video_ids)


import os
import json

PROGRESS_PATH = f"{BASE}/Project2data/raw/progress.json"   # tracks which search terms are done
CSV_PATH      = f"{BASE}/Project2data/raw/video_metadata.csv"


# ── Save progress after each search term completes
def save_progress(completed_queries: list):
    with open(PROGRESS_PATH, "w") as f:
        json.dump(completed_queries, f)            # writes the list of finished queries to disk
    print(f" Progress saved — {len(completed_queries)} queries done")


# ── Load progress so we can skip already-finished queries
def load_progress() -> list:
    if os.path.exists(PROGRESS_PATH):
        with open(PROGRESS_PATH, "r") as f:
            completed = json.load(f)
            print(f"📂 Resuming — {len(completed)} queries already done, skipping them")
            return completed
    return []



In [ ]:
SEARCH_TERMS = [
    ("Music",          "official music video 2024"),
    ("Music",          "official music video 2023"),
    ("Music",          "new song lyrics video 2024"),
    ("Gaming",         "gaming walkthrough gameplay 2024"),
    ("Gaming",         "gaming highlights best moments 2024"),
    ("Gaming",         "let's play full game 2023"),
    ("People & Blogs", "vlog daily life 2024"),
    ("People & Blogs", "day in my life vlog 2023"),
    ("People & Blogs", "storytime vlog 2024"),
    ("Entertainment",  "funny entertainment viral 2024"),
    ("Entertainment",  "comedy sketch viral video 2023"),
    ("Entertainment",  "reaction video entertainment 2024"),
    ("How-to",         "how to tutorial tips 2024"),
    ("How-to",         "diy tutorial beginner guide 2023"),
    ("How-to",         "cooking recipe tutorial 2024"),
    ("Education",      "learn explained education 2024"),
    ("Education",      "beginner course explained simply 2023"),
    ("Education",      "documentary explained history 2024"),
    ("Science & Tech", "science technology review 2024"),
    ("Science & Tech", "tech review unboxing 2023"),
    ("Science & Tech", "science experiment explained 2024"),
    ("Sports",         "sports highlights 2024"),
    ("Sports",         "best sports moments compilation 2023"),
    ("Sports",         "sports documentary 2024"),
]

# ── picks up where it left off
completed_queries = load_progress()
all_dfs = []

if os.path.exists(CSV_PATH):
    all_dfs.append(pd.read_csv(CSV_PATH))
    print(f"📂 Loaded existing CSV — {len(all_dfs[0])} rows already saved")

for category, query in SEARCH_TERMS:

    if query in completed_queries:
        print(f"⏭ Skipping already done: {query}")
        continue

    try:
        df = search_and_fetch(query, max_results=500)
        df["category"] = category
        all_dfs.append(df)

        final_df = pd.concat(all_dfs, ignore_index=True)
        final_df.to_csv(CSV_PATH, index=False)

        completed_queries.append(query)
        save_progress(completed_queries)

        print(f"✓ Done: {query} | Key 1: {units_used[0]} units | Key 2: {units_used[1]} units")

    except Exception as e:
        print(f"\n🛑 Stopped at: {query}")
        print(f"   Reason: {e}")
        print(f"   Key 1 used: {units_used[0]} / {DAILY_LIMIT}")
        print(f"   Key 2 used: {units_used[1]} / {DAILY_LIMIT}")
        print(f"   Resume tomorrow — progress saved ✓")
        break

if all_dfs:
    final_df = pd.concat(all_dfs, ignore_index=True)
    print(f"\n── Summary ──────────────────────────────")
    print(f"Total videos:  {len(final_df)}")
    print(f"Shorts:        {final_df['is_short'].sum()}")
    print(f"Regular:       {(~final_df['is_short']).sum()}")
    print(f"Queries done:  {len(completed_queries)} / {len(SEARCH_TERMS)}")

# EDA

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast


df = pd.read_csv(CSV_PATH)
print(f"Shape: {df.shape}")   # rows x columns

In [ ]:
# size, column names, types, null counts
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
df.head(5)

In [ ]:
# distribution of categories
print(df["category"].value_counts())

df["category"].value_counts().plot(kind="bar", figsize=(10,4), color="steelblue")
plt.title("Videos per category")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# shorts vs long form
print(df["is_short"].value_counts())

df["is_short"].value_counts().plot(kind="pie", autopct="%1.1f%%", figsize=(5,5),
                                    labels=["Regular", "Short"])
plt.title("Shorts vs Regular videos")
plt.show()

In [ ]:
print(df["category"].value_counts())

In [ ]:
# view count distribution

# raw distribution (very skewed)
df["view_count"].describe()

In [ ]:
# log transfomration view count

# log scale is much more readable for views
import numpy as np

df["log_views"] = np.log1p(df["view_count"])   # log1p handles 0 views safely

df["log_views"].hist(bins=50, figsize=(10,4), color="steelblue", edgecolor="white")
plt.title("Distribution of log(view count)")
plt.xlabel("log(views)")
plt.ylabel("Count")
plt.show()

In [ ]:
# view count by category
df.groupby("category")["view_count"].median().sort_values(ascending=False).plot(
    kind="bar", figsize=(10,4), color="coral")
plt.title("Median view count per category")
plt.ylabel("Median views")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# likes, comments vs. views

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(df["log_views"], np.log1p(df["like_count"]), alpha=0.3, s=5)
axes[0].set_title("Log views vs log likes")
axes[0].set_xlabel("log(views)")
axes[0].set_ylabel("log(likes)")

axes[1].scatter(df["log_views"], np.log1p(df["comment_count"]), alpha=0.3, s=5, color="coral")
axes[1].set_title("Log views vs log comments")
axes[1].set_xlabel("log(views)")
axes[1].set_ylabel("log(comments)")

plt.tight_layout()
plt.show()

In [ ]:
# thumbnail quality

print(df["thumbnail_quality"].value_counts())

df["thumbnail_quality"].value_counts().plot(kind="bar", figsize=(7,4), color="mediumpurple")
plt.title("Thumbnail quality levels downloaded")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# quick sumary stats per category

df.groupby("category")[["view_count","like_count","comment_count"]].agg(["mean","median","max"])

##### Note: Addressing view count issue. Some video took many years to get a high view count while other need time


In [ ]:
# nromalize view count

from datetime import datetime, timezone

In [ ]:
#  make sure published_at is datetime
df["published_at"] = pd.to_datetime(df["published_at"], utc=True)

# calculate age of each video in days
now = datetime.now(timezone.utc)                                    # today's date UTC
df["age_days"] = (now - df["published_at"]).dt.days                 # how many days since upload
df["age_days"] = df["age_days"].clip(lower=1)                       # avoiding division by zero for brand new videos

# views per day
df["views_per_day"] = df["view_count"] / df["age_days"]             # raw views per day

# log of views per day (still skewed without log)
df["log_views_per_day"] = np.log1p(df["views_per_day"])             # log smooths the skew

In [ ]:
# compare old vs new metric
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(np.log1p(df["view_count"]), bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("log(view_count) — biased by age")
axes[0].set_xlabel("log(views)")

axes[1].hist(df["log_views_per_day"], bins=50, color="coral", edgecolor="white")
axes[1].set_title("log(views_per_day) — age adjusted")
axes[1].set_xlabel("log(views/day)")

plt.tight_layout()
plt.show()

In [ ]:
# see the difference on real examples
df[["title", "published_at", "age_days", "view_count", "views_per_day"]]\
    .sort_values("views_per_day", ascending=False).head(10)

In [ ]:
# views per day by category
df.groupby("category")["views_per_day"].median().sort_values(ascending=False).plot(
    kind="bar", figsize=(10,4), color="coral")
plt.title("Median views per day by category (age-adjusted)")
plt.ylabel("Views per day")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# save updated dataframe with new columns back to CSV
df.to_csv(CSV_PATH, index=False)
print(f"Saved {len(df)} rows with new columns to CSV")
print(f"Columns now: {list(df.columns)}")